In [19]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "krupenye2017test")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Krupenye Kano et al 2017 MPI Database.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [20]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

# df.columns = map(str.lower, df.columns)
# df=df.applymap(lambda s: s.lower() if type(s) == str else s)

name_replace = [["Dasa","daza"],
        ["Luisa",	"luiza"],
        ["Bombari",	"bambari"],
        ["Natasha",	"natascha"],
        ["Swela baby","azibo"	]]

for x,y in name_replace:
    df['Subject'].replace(x, y, inplace=True, regex=True)

df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s) 
# df.columns

In [21]:
follow_up = df[['facility', 'subject', 'condition', 'first look',
       'target_duration', 'distractor_duration', 'look_diff']]
follow_up =follow_up.assign(experiment_name='2017_follow_up')
original = df[['facility', 'subject','condition.1','first look.1', 'target_duration.1',
       'distractor_duration.1', 'look_diff.1']]
original =original.assign(experiment_name='2016_original')
original.columns = original.columns.str.replace('.1', '', regex=True)

In [22]:
data_frames=[follow_up, original]
for index, x in enumerate(data_frames):
    x.rename(columns={"subject": "participant"}, inplace=True) ##standardize names for participants
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    x['study_id']="krupenye2017test"
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [23]:
fulldf['participant'].replace('kasai', 'kasai_II', inplace=True)
fulldf['participant'].unique()

array(['alex', 'bangolo', 'batak', 'daza', 'dokana', 'fimi', 'fraukje',
       'frederike', 'gemena', 'jeudi', 'kasai_II', 'kofi', 'kuno', 'lexi',
       'lobo', 'lome', 'luiza', 'padana', 'pini', 'raja', 'riet',
       'robert', 'suaq', 'tanah', 'yaro', 'yasa', 'bambari', 'frodo',
       'hatsuka', 'hope', 'ikela', 'iroha', 'jr', 'kisha', 'lenore',
       'loleta', 'louise', 'mizuki', 'natascha', 'natsuki', 'swela',
       'azibo', 'vijay', 'jahaga', 'jasongo', 'kara', 'sandra'],
      dtype=object)

In [24]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

comp_path_ape_info_2 = os.path.join(original_data_pathway, "non_mpi_participants_kano2018human.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)   
apedf_2_list =apedf_2.values.tolist()
for x,y,k in apedf_2_list:
    fulldf.loc[fulldf.participant == x, ['sex', 'species']] = y, k 
# fulldf= fulldf.merge(apedf_2,left_on='participant', right_on='name', how='left')
# fulldf.columns
fulldf.rename(columns={"first look": "first_look"}, inplace=True)

In [25]:

studyID_standardized=fulldf[['study_id','experiment_name', 'participant', 'sex', 'species','facility',
         'condition', 'first_look', 'target_duration',
       'distractor_duration', 'look_diff']]

comp_out_path_stand = os.path.join(out_pathway, 'krupenye2017test_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

##for one glossary
names = studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'krupenye2017test_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

# exp1 = fulldf[fulldf['experiment_name'] == '2017_follow_up']
# exp2 = fulldf[fulldf['experiment_name'] == '2016_original']

# experiments = [[exp1, 'krupenye2017test_exp1'],
#                 [ exp2, 'krupenye2017test_exp2']]

# for x,y in experiments:
#     x = x.dropna(axis=1, how='all')## drop empty rows/columns
#     comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
#     x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
#     ##glossaries
#     names = x.columns.tolist()
#     df = pd.DataFrame(names)
#     df = df.rename(columns={0: "column_name"})
#     df["description"] = ""
#     studyID_glossary=df[["column_name", "description"]]

    # comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
    # studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

